# Score the comparison models on the paper's 623 test tiles

**Inputs:** the DeepGlobe dataset, plus one Kaggle dataset or model holding the checkpoints:
every `best_<experiment>.pth` produced by `train_compare.ipynb` (v4 comes with the repository).
Settings: GPU T4, Internet on. Run All (about 10 minutes per model).

The notebook first re-measures v4 and **stops with an error** if it does not reproduce the paper
(TTA IoU 61.07%, single pass IoU 59.81%), so every other row is scored exactly like the paper.
Output: `/kaggle/working/compare_results.zip` (JSON, per tile CSV, LaTeX table rows).

In [ ]:
# Settings
PAPER_V4 = {"tta_raw": 0.61073, "single_raw": 0.59807}   # results.json of the paper
TOL = 0.002
VARIANTS = {"single": ["raw"], "tta": ["raw", "full"]}
OUT = "/kaggle/working/compare_results"

In [ ]:
# Code from the repository
import os, sys, glob, json, subprocess, shutil, zipfile
REPO_URL = "https://github.com/Dilraj07/Satellite-Imagery-based-Road-Network-Extraction.git"
REPO_REF = "main"          # a branch, or a commit hash to pin the exact code
REPO = "/kaggle/working/repo"
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", "https://" + UserSecretsClient().get_secret("GITHUB_TOKEN") + "@")
except Exception:
    pass
env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
if not os.path.isdir(REPO):
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", REPO_REF, url, REPO], env=env).returncode:
        if subprocess.run(["git", "clone", "-q", url, REPO], env=env).returncode:        # REPO_REF is a commit
            raise RuntimeError("Could not download the repository: make it public, or add the Kaggle secret GITHUB_TOKEN.")
        subprocess.run(["git", "-C", REPO, "checkout", "-q", REPO_REF], check=True)
else:
    if not subprocess.run(["git", "-C", REPO, "fetch", "-q", "--depth", "1", url, REPO_REF], env=env).returncode:
        subprocess.run(["git", "-C", REPO, "checkout", "-q", "FETCH_HEAD"], env=env)
subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", REPO_URL], check=True)   # no token in the saved output
V4 =os.path.join(REPO, "models", "best_model_v4.pth")   # the paper's model, included in the repository
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)
try:
    import skimage
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-image"], check=True)
sys.path.insert(0, os.path.join(REPO, "scripts", "experiments"))
import numpy as np, pandas as pd, torch
import exp_data as D
import exp_torch as T
DEVICE = torch.device("cuda")
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0))

In [ ]:
# Test tiles and canopy threshold (as in the paper)
os.makedirs(OUT, exist_ok=True)
DG = D.find_deepglobe()
test_items = D.deepglobe_items(DG, D.deepglobe_split(DG)["test"])
VEG_T = D.otsu_vegetation_threshold(test_items)
print(len(test_items), "test tiles | vegetation threshold", VEG_T, "(paper: 0.1191)")
assert abs(VEG_T - 0.119140625) < 1e-6

In [ ]:
# Checkpoints: v4 plus one file per experiment (.pth, browser-renamed .zip, or unpacked Kaggle dir)
found = {}
for path in T.find_checkpoints("/kaggle/input", prefix="best_"):
    base = os.path.basename(path)
    if base.startswith("best_model_v4") or "train_state" in base:
        continue
    ck = torch.load(path, map_location="cpu", weights_only=False)
    if not isinstance(ck, dict) or "experiment" not in ck:
        continue
    name = ck["experiment"]
    if ck["spec"].get("dataset", "deepglobe") != "deepglobe":
        continue
    assert ck.get("test_ids_sha1") == D.TEST_SHA1, f"{path}: trained on a different split"
    if name not in found or ck["val_iou"] > found[name][1]["val_iou"]:
        found[name] = (path, {k: v for k, v in ck.items() if k != "model_state_dict"})
found["v4"] = (V4, {})
order = ["v4"] + [e for e in T.EXPERIMENTS if e in found]
for n in order:
    print(f"{n:20s} {found[n][0]}  epoch {found[n][1].get('epoch', 158)}")

In [ ]:
# Evaluate
RECORDS, PARAMS = {}, {}
for n in order:
    model, meta = T.load_any_checkpoint(found[n][0], DEVICE)
    PARAMS[n] = T.n_params(model)
    print(f"{n}: {PARAMS[n]:,} parameters")
    RECORDS[n] = T.evaluate(model, test_items, VARIANTS, VEG_T, DEVICE)
    del model; torch.cuda.empty_cache()
    if n == "v4":
        got = {v: D.summarize(D.count_matrix(RECORDS[n], v))["iou"] for v in PAPER_V4}
        print("v4 reproduction:", got, "paper:", PAPER_V4)
        assert all(abs(got[v] - PAPER_V4[v]) <= TOL for v in PAPER_V4), "v4 does not reproduce the paper; stop here"
        print("V4 REPRODUCED")

In [ ]:
def table(rows):
    """rows: [(name, params, summary dict)] -> printed table and LaTeX lines."""
    keys = [("precision", "P"), ("recall", "R"), ("f1", "F1"), ("iou", "IoU"),
            ("relaxed_f1", "Rel. F1"), ("cldice", "clDice"), ("canopy_recall", "Canopy R")]
    df = pd.DataFrame([{"model": n, "params (M)": round(p / 1e6, 2) if p else None,
                        **{h: round(100 * s[k], 1) for k, h in keys},
                        "parts": int(round(s["median_components_per_tile"]))} for n, p, s in rows])
    print(df.to_string(index=False))
    latex = [f"{n} & {p / 1e6:.2f} & " + " & ".join(f"{100 * s[k]:.1f}" for k, _ in keys)
             + f" & {s['median_components_per_tile']:.0f} \\\\" for n, p, s in rows]
    return df, latex

In [ ]:
# Results
SUMMARY = {n: {f"{s}_{v}": D.summary(RECORDS[n], f"{s}_{v}") for s, vl in VARIANTS.items() for v in vl} for n in order}
TESTS = {}
for n in order:
    ref = "v4" if not n.startswith("abl_") else ("abl_full" if n != "abl_full" else None)
    if n == "v4" or ref is None or ref not in RECORDS:
        continue
    for var in ("tta_raw", "tta_full"):
        TESTS[f"{ref}_vs_{n}_{var}"] = D.paired_test(RECORDS[ref], RECORDS[n], var)
for var in ("tta_raw", "tta_full", "single_raw"):
    print(f"\n=== {var} ===")
    _, latex = table([(n, PARAMS[n], SUMMARY[n][var]) for n in order])
    open(f"{OUT}/table_{var}.tex", "w").write("\n".join(latex) + "\n")
print("\nPaired Wilcoxon tests on per tile IoU (positive mean_diff: the first model is better):")
for k, t in TESTS.items():
    print(f"  {k:45s} mean diff {100 * t['mean_diff']:+.2f} points | better on {100 * t['frac_tiles_better']:.0f}% of tiles | p = {t['wilcoxon_p']}")

In [ ]:
# Save
rows = []
for n in order:
    for var in SUMMARY[n]:
        for tile, m in D.per_tile(RECORDS[n], var).items():
            rows.append({"model": n, "variant": var, "tile": tile, **m})
pd.DataFrame(rows).to_csv(f"{OUT}/per_tile_metrics.csv", index=False)
json.dump({"summary": SUMMARY, "paired_tests": TESTS, "params": PARAMS, "vegetation_threshold": VEG_T,
           "checkpoints": {n: {"path": found[n][0], **{k: v for k, v in found[n][1].items()
                                                         if isinstance(v, (int, float, str, dict, list))}}
                           for n in order}},
          open(f"{OUT}/compare_results.json", "w"), indent=1, default=str)
shutil.make_archive("/kaggle/working/compare_results", "zip", OUT)
print("saved /kaggle/working/compare_results.zip")